# Proyek Sains Data - Ekstraksi Fitur Citra

Notebook ini melakukan ekstraksi fitur warna (RGB) dan fitur tekstur GLCM dari dataset citra pada folder `foto-psd`:
- **Fitur Warna (RGB - Slide 9-11)**:
  - Rata-rata (Mean): `mean_r`, `mean_g`, `mean_b`
  - Standar Deviasi (Standard Deviation): `std_r`, `std_g`, `std_b`
  - Kemiringan Distribusi (Skewness): `skew_r`, `skew_g`, `skew_b`
- **Fitur Tekstur GLCM (Slide 19 & 20)**:
  - Kontras (Contrast - Pers. 6): `glcm_contrast`
  - Entropi (Entropy - Pers. 5): `glcm_entropy`
  - Korelasi (Correlation - Pers. 7): `glcm_correlation`
  - Energi (Energy / ASM - Pers. 8): `glcm_energy`
  - Homogenitas (Homogeneity - Pers. 9): `glcm_homogeneity`
- **Label / Target**:
  - `kelas`: Diambil otomatis dari nama subfolder (`rapi`, `berantakan`)


In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from scipy.stats import skew
from skimage.feature import graycomatrix, graycoprops

# 1. Path folder dataset citra terbaru
folder_dataset = 'foto-psd'

# Format file citra yang didukung
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

# List untuk menampung hasil ekstraksi fitur
data_fitur = []

# Precompute matriks indeks untuk perhitungan GLCM (Slide 19 & 20)
i_idx, j_idx = np.indices((256, 256))
diff_abs = 1.0 + np.abs(i_idx - j_idx)
diff_sq = (i_idx - j_idx)**2

# 2. Iterasi setiap sub-folder (kelas) dan file gambar di dalam folder foto-psd
for root, dirs, files in os.walk(folder_dataset):
    for file in files:
        if file.lower().endswith(valid_extensions):
            file_path = os.path.join(root, file)
            
            # Kelas didapat dari nama folder tempat citra berada (misal: 'rapi', 'berantakan')
            kelas = os.path.basename(root)
            
            # Membaca citra menggunakan OpenCV (format bawaan: BGR)
            img_bgr = cv2.imread(file_path)
            
            # Fallback jika cv2.imread mengembalikan None (misal handling path tertentu)
            if img_bgr is None:
                img_bgr = cv2.imdecode(np.fromfile(file_path, dtype=np.uint8), cv2.IMREAD_COLOR)
                
            if img_bgr is not None:
                # -----------------------------------------------------------------
                # A. FITUR WARNA (RGB) - Slide 9, 10, 11
                # -----------------------------------------------------------------
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                
                # Mean (Rata-rata) tiap kanal warna
                mean_r, mean_g, mean_b = img_rgb.mean(axis=(0, 1))
                
                # Standar Deviasi tiap kanal warna
                std_r, std_g, std_b = img_rgb.std(axis=(0, 1))
                
                # Skewness (Kemencengan distribusi) tiap kanal warna
                skew_r, skew_g, skew_b = skew(img_rgb.reshape(-1, 3), axis=0)
                
                # -----------------------------------------------------------------
                # B. FITUR TEKSTUR GLCM - Slide 19 & 20
                # -----------------------------------------------------------------
                # Konversi ke Grayscale 8-bit untuk analisis spasial tekstur
                img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
                
                # Menghitung matriks GLCM ternormalisasi (jarak d=1, sudut theta=0 derajat)
                glcm = graycomatrix(img_gray, distances=[1], angles=[0], levels=256, symmetric=False, normed=True)
                p = glcm[:, :, 0, 0]
                
                # 1. Kontras (Slide 19, Pers. 6): sum (i - j)^2 * P(i, j)
                glcm_contrast = float(np.sum(diff_sq * p))
                
                # 2. Entropi (Slide 19, Pers. 5): -sum P(i, j) * log2(P(i, j))
                mask = p > 0
                glcm_entropy = float(-np.sum(p[mask] * np.log2(p[mask])))
                
                # 3. Korelasi (Slide 20, Pers. 7)
                glcm_correlation = float(graycoprops(glcm, 'correlation')[0, 0])
                
                # 4. Energi / ASM (Slide 20, Pers. 8): sum P(i, j)^2
                glcm_energy = float(np.sum(p**2))
                
                # 5. Homogenitas (Slide 20, Pers. 9): sum P(i, j) / (1 + |i - j|)
                glcm_homogeneity = float(np.sum(p / diff_abs))
                
                # -----------------------------------------------------------------
                # Simpan seluruh fitur ke list
                # -----------------------------------------------------------------
                data_fitur.append({
                    'nama_file': file,
                    'mean_r': float(mean_r),
                    'mean_g': float(mean_g),
                    'mean_b': float(mean_b),
                    'std_r': float(std_r),
                    'std_g': float(std_g),
                    'std_b': float(std_b),
                    'skew_r': float(skew_r),
                    'skew_g': float(skew_g),
                    'skew_b': float(skew_b),
                    'glcm_contrast': glcm_contrast,
                    'glcm_entropy': glcm_entropy,
                    'glcm_correlation': glcm_correlation,
                    'glcm_energy': glcm_energy,
                    'glcm_homogeneity': glcm_homogeneity,
                    'kelas': kelas
                })

# 3. Membuat DataFrame dari hasil ekstraksi
df = pd.DataFrame(data_fitur)

print(f"Total citra yang berhasil diekstraksi: {len(df)}")
df.head()


Total citra yang berhasil diekstraksi: 194


,nama_file,mean_r,mean_g,mean_b,std_r,std_g,std_b,skew_r,skew_g,skew_b,glcm_contrast,glcm_entropy,glcm_correlation,glcm_energy,glcm_homogeneity,kelas
0,24-003 Berantakan 2.jpg,108.554706,101.407629,96.538574,56.155452,54.829354,53.918995,0.814838,0.826725,0.791602,71.908726,10.724201,0.987758,0.002500,0.608047,berantakan
1,24-003 Berantakan 3.jpg,149.420848,129.994709,112.171947,74.963486,71.982165,65.608857,-0.195272,-0.084969,-0.051042,160.542981,12.771686,0.984264,0.000259,0.327691,berantakan
2,24-003 berantakan.jpg,144.580847,109.836210,85.609472,72.227316,59.930651,55.883418,-0.028776,0.283462,0.659771,149.946906,12.517000,0.979332,0.000324,0.338605,berantakan
3,24-006 Berantakan 1.jpg,120.539929,112.681118,109.410009,54.881074,52.943897,52.596739,-0.479489,-0.259841,-0.174987,41.878633,10.706465,0.992559,0.001375,0.554793,berantakan
4,24-006 Berantakan 2.jpg,126.631404,119.915821,116.691646,55.209520,51.268704,49.920694,-0.741969,-0.557391,-0.411285,20.982264,10.025935,0.996143,0.002165,0.632239,berantakan


In [2]:
# Menampilkan informasi ringkas dataset dan jumlah sampel per kelas
print("Informasi Dataset:")
df.info()

print("\nJumlah Sampel per Kelas:")
print(df['kelas'].value_counts())

Informasi Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 194 entries, 0 to 193
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   nama_file         194 non-null    object 
 1   mean_r            194 non-null    float64
 2   mean_g            194 non-null    float64
 3   mean_b            194 non-null    float64
 4   std_r             194 non-null    float64
 5   std_g             194 non-null    float64
 6   std_b             194 non-null    float64
 7   skew_r            194 non-null    float64
 8   skew_g            194 non-null    float64
 9   skew_b            194 non-null    float64
 10  glcm_contrast     194 non-null    float64
 11  glcm_entropy      194 non-null    float64
 12  glcm_correlation  194 non-null    float64
 13  glcm_energy       194 non-null    float64
 14  glcm_homogeneity  194 non-null    float64
 15  kelas             194 non-null    object 
dtypes: float64(14), object(2)

In [3]:
# Menyimpan hasil ekstraksi fitur ke file CSV
output_csv = 'hasil_ekstraksi_fitur.csv'
try:
    df.to_csv(output_csv, index=False)
    print(f"Hasil ekstraksi fitur berhasil disimpan ke '{output_csv}'")
except PermissionError:
    # Jika file utama sedang dibuka di aplikasi lain (seperti Microsoft Excel)
    alt_csv = 'hasil_ekstraksi_fitur_terbaru.csv'
    df.to_csv(alt_csv, index=False)
    print(f"Perhatian: '{output_csv}' sedang dibuka oleh aplikasi lain (misal: Microsoft Excel).")
    print(f"Hasil ekstraksi berhasil disimpan ke '{alt_csv}'. Tutup Excel jika ingin menimpa file utama.")

Hasil ekstraksi fitur berhasil disimpan ke 'hasil_ekstraksi_fitur.csv'
